<h1 style="padding-top: 25px; padding-bottom: 25px; text-align: left; padding-left: 10px; background-color: #F7F1E8; border-bottom: 4px solid #D52B1E; color: #1E1C24;"> <img style="float: left; padding-right: 10px;" src="https://raw.githubusercontent.com/pavlosprotopapas/elelem-chile-2026/main/assets/epoch_tour_mark.png" height="50px"> <a href="https://pavlosprotopapas.github.io/elelem-chile-2026/" target="_blank"><strong><font color="#1E1C24">ELELEM · Epoch Tour Chile 2026 — From LLMs to Agentic Systems</font></strong></a></h1>

# **<font color="#D52B1E">RAG lab — answer questions from your own documents</font>**

**ELELEM Chile 2026** · Santiago, 22–24 October 2026<br/>
**Day 1**<br/>
**Instructors:** Pavlos Protopapas (Harvard University) · Guillermo Cabrera-Vives (Universidad de Concepción)<br/>

<hr style="height:2pt">

**Time:** 30 minutes · **Runs in:** Google Colab — nothing to install · **Needs:** `OPENAI_API_KEY` in Colab secrets (🔑, from the pre-work)

This is the Colab version of the [llm-rag tutorial](https://github.com/dlops-io/llm-rag), which normally runs in Docker on GCP. Same corpus — eleven books about cheese — and the same pipeline:

1. 📦 **Chunk** the books into small, overlapping pieces
2. 🔢 **Embed** each chunk into a vector
3. 🗄️ **Load** the vectors into ChromaDB
4. 🔍 **Query** — retrieve the chunks closest to a question
5. 💬 **Chat** — give the retrieved chunks to the LLM and let it answer from them

**Run every cell top to bottom first (≈5 min).** Then do the one TODO at the end.

## 0 · Setup

In [ ]:
%pip install -q chromadb langchain-text-splitters openai

In [ ]:
import os, json, urllib.parse, urllib.request
import chromadb
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 256          # small vectors are enough here, and cheaper to store
CHAT_MODEL = "gpt-4o-mini"         # any current chat model works; change it here
print("✅ setup done")

## 1 · Get the books

Plain-text books from Project Gutenberg, plus one short book that no LLM has seen in training: *Tolminc Cheese* by Pavlos Protopapas (2024). Keep an eye on that one.

In [ ]:
BOOKS = {
    "Cheese and its economical uses in the diet": ("C. F. Langworthy and Caroline Louisa Hunt", 2023),
    "Cottage Cheese Recipe Book": ("Milk Industry Foundation", 2021),
    "Dairying exemplified, or, The business of cheese-making": ("J. Twamley", 2023),
    "Hand-book on cheese making": ("George E. Newell", 2023),
    "Hints on cheese-making, for the dairyman, the factoryman, and the manufacturer": ("T. D. Curtis", 2013),
    "The Book of Cheese": ("Charles Thom and W. W. Fisk", 2012),
    "The book of The Cheese Being traits and stories of Ye Olde Cheshire Cheese": ("Thomas Wilson Reid", 2023),
    "The Complete Book of Cheese": ("Bob Brown", 2024),
    "Theres Pippins and Cheese to Come": ("Charles S. Brooks", 2003),
    "Womans Institute Library of Cookery. Volume 2_ Milk, Butter and Cheese Eggs Vegetables": ("Woman's Institute of Domestic Arts and Sciences", 2006),
    "Tolminc Cheese": ("Pavlos Protopapas", 2024),
}
BASE = "https://raw.githubusercontent.com/dlops-io/llm-rag/main/input-datasets/books/"

texts = {}
for title in BOOKS:
    url = BASE + urllib.parse.quote(title + ".txt")
    texts[title] = urllib.request.urlopen(url).read().decode("utf-8", errors="ignore")
    print(f"{len(texts[title]):>8,} chars  {title}")

## 2 · Chunk

Two splitters from the lecture:
- **character split** — cut every 350 characters, with a 20-character overlap so ideas are not cut in half;
- **recursive split** — try to cut at paragraph, then line, then sentence boundaries, and only fall back to characters.

Compare what one chunk looks like with each.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

def chunk(method="char-split"):
    if method == "char-split":
        splitter = CharacterTextSplitter(chunk_size=350, chunk_overlap=20, separator="", strip_whitespace=False)
    else:
        splitter = RecursiveCharacterTextSplitter(chunk_size=350)
    chunks = []
    for title, text in texts.items():
        author, year = BOOKS[title]
        for i, piece in enumerate(splitter.split_text(text)):
            chunks.append({"id": f"{title[:40]}-{i}", "text": piece, "book": title, "author": author, "year": year})
    return chunks

char_chunks = chunk("char-split")
rec_chunks = chunk("recursive-split")
print(len(char_chunks), "char-split chunks |", len(rec_chunks), "recursive-split chunks\n")
print("char-split  →", repr(char_chunks[1000]["text"]), "\n")
print("recursive   →", repr(rec_chunks[1000]["text"]))

## 3 · Embed

Each chunk becomes a vector of 256 numbers. Chunks with similar meaning end up close together. We send them in batches; the whole corpus costs a few cents.

In [ ]:
def embed(texts_, batch_size=500):
    vectors = []
    for i in range(0, len(texts_), batch_size):
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=texts_[i:i + batch_size], dimensions=EMBEDDING_DIMENSION)
        vectors += [d.embedding for d in resp.data]
        print(f"embedded {min(i + batch_size, len(texts_))}/{len(texts_)}", end="\r")
    return vectors

chunks = char_chunks                      # TODO (end of lab): try rec_chunks
vectors = embed([c["text"] for c in chunks])
print("\nvector length:", len(vectors[0]), "| first 5 numbers:", [round(x, 3) for x in vectors[0][:5]])

## 4 · Load into a vector database

ChromaDB stores each vector with its text and metadata (book, author, year) and can find nearest neighbours fast. Here it runs in memory inside Colab; in the Docker version it is a separate server.

In [ ]:
db = chromadb.Client()
collection = db.get_or_create_collection("cheese", metadata={"hnsw:space": "cosine"})
for i in range(0, len(chunks), 1000):
    batch = chunks[i:i + 1000]
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=vectors[i:i + 1000],
        metadatas=[{"book": c["book"], "author": c["author"], "year": c["year"]} for c in batch],
    )
print("chunks in the collection:", collection.count())

## 5 · Query

Embed the question with the same model, then ask Chroma for the closest chunks. Metadata filters (`where`) and keyword filters (`where_document`) narrow the search.

In [ ]:
def retrieve(question, n_results=10, **filters):
    q = client.embeddings.create(model=EMBEDDING_MODEL, input=[question], dimensions=EMBEDDING_DIMENSION).data[0].embedding
    return collection.query(query_embeddings=[q], n_results=n_results, **filters)

question = "How is cheese made?"
res = retrieve(question, n_results=5)
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"[{dist:.3f}] {meta['book'][:45]}\n   {doc[:160]!r}\n")

# The same search, but only inside one book and only chunks that mention "milk":
res = retrieve(question, n_results=3, where={"book": "The Complete Book of Cheese"}, where_document={"$contains": "milk"})
print("filtered:", [d[:80] for d in res["documents"][0]])

## 6 · Chat — retrieval-augmented generation

The system prompt tells the model to answer **only** from the chunks it is given. First ask without retrieval, then with it.

In [ ]:
SYSTEM_INSTRUCTION = """You are an AI assistant specialized in cheese knowledge. Answer only from the text chunks provided.
If the chunks do not contain enough information, say so. If chunks contradict each other, say so and explain both views.
Be concise. If asked about topics unrelated to cheese, politely steer back to cheese."""

def ask(question, use_rag=True, n_results=10, show_sources=True):
    if use_rag:
        res = retrieve(question, n_results=n_results)
        context = "\n\n".join(res["documents"][0])
        prompt = f"{question}\n\nText chunks:\n{context}"
        system = SYSTEM_INSTRUCTION
    else:
        prompt, system = question, "You are a helpful assistant."
    answer = client.chat.completions.create(
        model=CHAT_MODEL, messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    ).choices[0].message.content
    print(("RAG" if use_rag else "NO RAG"), "→", answer)
    if use_rag and show_sources:
        print("\nsources:", sorted({m["book"][:40] for m in res["metadatas"][0]}))
    print("-" * 80)

q = "How is cheese made in Tolminc, and who wrote about it?"
ask(q, use_rag=False)
ask(q, use_rag=True)

## ✏️ TODO — one experiment (10 minutes)

Pick **one** knob, change it, re-run from the cell where it lives, and ask the same two questions again:

```python
ask("How is cheese made in Tolminc, and who wrote about it?")
ask("What is the difference between hard and soft cheese?")
```

| Knob | Where | Try |
|---|---|---|
| Chunking | step 3: `chunks = char_chunks` | `rec_chunks` (then re-run steps 3–6) |
| How many chunks the LLM sees | `ask(..., n_results=10)` | `2` and `30` |
| The system prompt | step 6 | delete the "answer only from the chunks" line |

Write down in one sentence: **what changed in the answer, and why?** We will collect a few answers in the room.

*Going further (optional):* the Docker version adds **semantic chunking** and an **agent** that decides which retrieval tool to call — see the [llm-rag repo](https://github.com/dlops-io/llm-rag). You will build agents tomorrow.